In [0]:
# =============================================================
# CELDA 1: GOLD — KPIs + Resumen de Mercado Robusto
# =============================================================
import json
import boto3
import re
import unicodedata
from pyspark.sql import functions as F, Window
from pyspark.sql.types import StringType

# Credenciales: Databricks secrets (producción) → archivo local (desarrollo)
try:
    config = {
        "aws_access_key": dbutils.secrets.get(scope="aws", key="access_key"),
        "aws_secret_key": dbutils.secrets.get(scope="aws", key="secret_key"),
    }
except Exception:
    try:
        with open("aws_secrets.json", "r") as f:
            config = json.load(f)
    except FileNotFoundError:
        raise RuntimeError("No se encontraron credenciales AWS.")

BUCKET = config.get("bucket_name", "bronce-scrap-date")

# ── S3_OPTS: NO incluir fs.s3a.impl — no compatible con Databricks Serverless ──
S3_OPTS = {
    "fs.s3a.access.key": config["aws_access_key"],
    "fs.s3a.secret.key": config["aws_secret_key"],
    "fs.s3a.endpoint":   "s3.amazonaws.com",
}

s3_check = boto3.client(
    "s3",
    aws_access_key_id=config["aws_access_key"],
    aws_secret_access_key=config["aws_secret_key"],
)

# ── Constantes de analítica (usadas en Celda 2) ──
MIN_ANALYTICS_OBS = {
    "market":  50,
    "city":    30,
    "comuna":  12,
    "sector":   8,
}

# ── UDFs Geográficas desde src.geo.sector_mapping ──
import sys
import os
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
from src.geo.sector_mapping import get_spark_udfs, normalize_text
assign_comuna_udf, extract_sector_udf, canonicalize_city_udf = get_spark_udfs()

from pyspark.sql.types import StringType
from pyspark.sql import functions as F
from src.geo.sector_mapping import normalize_text
normalize_text_udf = F.udf(normalize_text, StringType())

def build_market_analytics(df_input, level_name, group_cols, min_obs):
    analytics = (
        df_input.groupBy(*(group_cols + ["tipo_inmueble"]))
        .agg(
            F.count("*").alias("market_n"),
            F.expr("percentile_approx(precio_num, 0.5)").alias("precio_mediano"),
            F.expr("percentile_approx(area_m2, array(0.25, 0.5, 0.75))").alias("area_quantiles"),
            F.expr("percentile_approx(precio_m2, array(0.05, 0.10, 0.25, 0.5, 0.75, 0.90, 0.95))").alias("precio_m2_quantiles"),
            F.round(F.avg(F.coalesce(F.col("num_portales"), F.lit(1))), 2).alias("promedio_portales"),
            F.round(F.avg(F.coalesce(F.col("dispersion_pct_grupo"), F.lit(0.0))), 1).alias("dispersion_promedio_pct"),
            F.round(F.avg(F.when(F.coalesce(F.col("num_portales"), F.lit(1)) > 1, F.lit(1.0)).otherwise(F.lit(0.0))) * 100, 1).alias("pct_multiportal"),
        )
        .filter(F.col("market_n") >= min_obs)
        .withColumn("analytics_level", F.lit(level_name))
    )
    if "city_token" not in group_cols:
        analytics = analytics.withColumn("city_token", F.lit("otra_ciudad"))
    if "comuna_mercado" not in group_cols:
        analytics = analytics.withColumn("comuna_mercado", F.lit("comuna_otra"))
    if "sector_mercado" not in group_cols:
        analytics = analytics.withColumn("sector_mercado", F.lit("sector_otra"))
    analytics = (
        analytics
        .withColumn("area_p25",         F.col("area_quantiles").getItem(0))
        .withColumn("area_mediana",      F.col("area_quantiles").getItem(1))
        .withColumn("area_p75",         F.col("area_quantiles").getItem(2))
        .withColumn("precio_m2_p05",    F.col("precio_m2_quantiles").getItem(0))
        .withColumn("precio_m2_p10",    F.col("precio_m2_quantiles").getItem(1))
        .withColumn("precio_m2_p25",    F.col("precio_m2_quantiles").getItem(2))
        .withColumn("precio_m2_mediano",F.col("precio_m2_quantiles").getItem(3))
        .withColumn("precio_m2_p75",    F.col("precio_m2_quantiles").getItem(4))
        .withColumn("precio_m2_p90",    F.col("precio_m2_quantiles").getItem(5))
        .withColumn("precio_m2_p95",    F.col("precio_m2_quantiles").getItem(6))
        .withColumn("precio_m2_iqr",    F.col("precio_m2_p75") - F.col("precio_m2_p25"))
        .withColumn("lower_bound_ref",
            F.greatest(F.lit(0.0), F.col("precio_m2_p10") - F.col("precio_m2_iqr") * F.lit(1.5)))
        .withColumn("upper_bound_ref",
            F.col("precio_m2_p90") + F.col("precio_m2_iqr") * F.lit(1.5))
        .withColumn("market_quality_score", F.round(F.least(F.lit(100.0),
            (F.log1p(F.col("market_n")) / F.log(F.lit(201.0))) * F.lit(60.0)
            + F.coalesce(F.col("pct_multiportal"), F.lit(0.0)) * F.lit(0.25)
            + (F.lit(100.0) - F.least(F.coalesce(F.col("dispersion_promedio_pct"), F.lit(100.0)), F.lit(100.0))) * F.lit(0.15)
        ), 1))
        .withColumn("market_key",
            F.concat_ws("__", "analytics_level", "city_token", "comuna_mercado", "sector_mercado", "tipo_inmueble"))
        .drop("area_quantiles", "precio_m2_quantiles", "precio_m2_iqr")
    )
    return analytics.select(
        "analytics_level", "market_key", "city_token", "comuna_mercado", "sector_mercado",
        "tipo_inmueble", "market_n", "precio_mediano",
        "area_p25", "area_mediana", "area_p75",
        "precio_m2_p05", "precio_m2_p10", "precio_m2_p25", "precio_m2_mediano",
        "precio_m2_p75", "precio_m2_p90", "precio_m2_p95",
        "lower_bound_ref", "upper_bound_ref",
        "promedio_portales", "dispersion_promedio_pct", "pct_multiportal", "market_quality_score",
    )



# ── Leer Silver Deduped ──
try:
    s3_check.list_objects_v2(
        Bucket=BUCKET, Prefix="silver/master_deduped/_delta_log/", MaxKeys=1
    )["Contents"]
    ruta_silver = f"s3a://{BUCKET}/silver/master_deduped/"
    print("📖 Leyendo Silver DEDUPED (cross-portal)")
except (KeyError, Exception):
    ruta_silver = f"s3a://{BUCKET}/silver/master_inmuebles/"
    print("📖 Leyendo Silver original (dedup no ejecutado aún)")

reader = spark.read.format("delta")
for k, v in S3_OPTS.items():
    reader = reader.option(k, v)
df_silver = reader.load(ruta_silver)

# ── Dedup intra-portal: mantener solo el snapshot más reciente por inmueble ──
# Sin esto, cada corrida del scraper acumula filas del mismo listing,
# generando duplicados visibles en Gold (ej. Flandes ×5, Calarcá ×5).
_n_before_dedup = df_silver.count()
print(f"   {_n_before_dedup} registros (pre-dedup)")
print(f"   Columnas: {df_silver.columns}")

_dedup_order_cols = [F.desc("fecha_extraccion")]
if "data_completeness" in df_silver.columns:
    _dedup_order_cols.append(F.desc("data_completeness"))

# ── Dedup intra-portal (Fuzzy): mantener solo el perfil físico más reciente ──
# Agrupamos por lo que define al inmueble, ignorando el ID original que suele cambiar.
_fuzzy_keys = ["fuente", "city_token", "tipo_inmueble", "precio_num", "area_m2", "habitaciones", "banos"]
# ── Dedup intra-portal (Fuzzy): mantener solo el perfil físico más reciente ──
# Agrupamos por lo que define al inmueble, ignorando el ID original que suele cambiar.
_fuzzy_keys = ["fuente", "city_token", "tipo_inmueble", "precio_num", "area_m2", "habitaciones", "banos"]
_w_latest = Window.partitionBy(*_fuzzy_keys).orderBy(F.desc("fecha_extraccion"))

_n_before_dedup_fuzzy = df_silver.count()
df_silver = (
    df_silver
    .withColumn("_dedup_rank", F.row_number().over(_w_latest))
    .filter(F.col("_dedup_rank") == 1)
    .drop("_dedup_rank")
)
_n_after_dedup_fuzzy = df_silver.count()
print(f"🧹 Dedup Fuzzy: {_n_before_dedup_fuzzy:,} -> {_n_after_dedup_fuzzy:,} inmuebles únicos.")

# ── Leer price intelligence si existe ──
try:
    s3_check.list_objects_v2(
        Bucket=BUCKET, Prefix="gold/price_intelligence/_delta_log/", MaxKeys=1
    )["Contents"]
    reader_intel = spark.read.format("delta")
    for k, v in S3_OPTS.items():
        reader_intel = reader_intel.option(k, v)
    df_intel = reader_intel.load(f"s3a://{BUCKET}/gold/price_intelligence/")
    print("💰 Price intelligence disponible")
except (KeyError, Exception):
    df_intel = None
    print("ℹ️ Price intelligence no disponible todavía")

# ── Gold base: filtros + KPIs + geolocalización ──
df_gold_base = (
    df_silver
    .filter(
        F.col("precio_num").isNotNull() & (F.col("precio_num") > 0) &
        F.col("area_m2").isNotNull() & (F.col("area_m2") >= 20) & (F.col("area_m2") <= 800)
    )
    .withColumn("precio_m2", F.round(F.col("precio_num") / F.col("area_m2"), 0))
    .withColumn("city_token", canonicalize_city_udf(F.coalesce(F.col("city_token"), F.lit("otra_ciudad"))))
    .withColumn("tipo_inmueble", F.coalesce(F.col("tipo_inmueble"), F.lit("otro")))
    .withColumn("_raw_norm", F.lower(F.trim(F.col("ubicacion_raw"))))
    .withColumn("zona_mercado", F.coalesce(
        F.when(F.length(F.col("_raw_norm")) > (F.length(F.col("city_token")) + F.lit(3)), F.col("_raw_norm")),
        F.when(F.length(F.col("ubicacion_norm")) > 4, F.col("ubicacion_norm")),
        F.when(F.length(F.col("_raw_norm")) > 4, F.col("_raw_norm")),
        F.concat_ws(" | ", F.col("city_token"), F.col("tipo_inmueble"))
    ))
    .drop("_raw_norm")
    .withColumn("ubicacion_limpia", normalize_text_udf(F.concat_ws(" | ", F.col("zona_mercado"), F.coalesce(F.col("titulo"), F.lit("")))))
    .withColumn("comuna_mercado", assign_comuna_udf(F.col("city_token"), F.col("ubicacion_limpia")))
    .withColumn("sector_mercado_raw", extract_sector_udf(F.col("city_token"), F.col("comuna_mercado"), F.col("ubicacion_limpia")))
    .withColumn("market_segment", F.concat_ws("__", F.col("city_token"), F.col("tipo_inmueble")))
    .withColumn("gold_processed_at", F.current_timestamp())
)

# ── Filtro por precio_m2 dentro de segmento ──
segment_stats = (
    df_gold_base.groupBy("market_segment")
    .agg(
        F.count("*").alias("segment_n"),
        F.expr("percentile_approx(precio_m2, 0.05)").alias("pm2_p05_segment"),
        F.expr("percentile_approx(precio_m2, 0.95)").alias("pm2_p95_segment"),
    )
)
global_pm2 = df_gold_base.agg(
    F.expr("percentile_approx(precio_m2, 0.02)").alias("pm2_p02_global"),
    F.expr("percentile_approx(precio_m2, 0.98)").alias("pm2_p98_global"),
).first()

df_gold = (
    df_gold_base
    .join(segment_stats, "market_segment", "left")
    .withColumn("pm2_lower_bound",
        F.when(F.col("segment_n") >= 15, F.col("pm2_p05_segment")).otherwise(F.lit(global_pm2["pm2_p02_global"])))
    .withColumn("pm2_upper_bound",
        F.when(F.col("segment_n") >= 15, F.col("pm2_p95_segment")).otherwise(F.lit(global_pm2["pm2_p98_global"])))
    .filter(F.col("precio_m2").between(F.col("pm2_lower_bound"), F.col("pm2_upper_bound")))
    .drop("pm2_lower_bound", "pm2_upper_bound", "pm2_p05_segment", "pm2_p95_segment")
)

# ── Colapsar sector_mercado: mínimo 12 obs o usar comuna ──
sector_counts = (
    df_gold.groupBy("city_token", "sector_mercado_raw")
    .agg(F.count("*").alias("sector_n_raw"))
)
df_gold = (
    df_gold
    .join(sector_counts, ["city_token", "sector_mercado_raw"], "left")
    .withColumn("sector_mercado",
        F.when(F.coalesce(F.col("sector_n_raw"), F.lit(0)) >= 12, F.col("sector_mercado_raw"))
         .when(F.col("comuna_mercado") != F.lit("comuna_otra"), F.col("comuna_mercado"))
         .otherwise(F.lit("sector_otra")))
    .drop("sector_mercado_raw", "sector_n_raw")
)

n_gold = df_gold.count()
n_con_barrio = df_gold.filter(F.col("comuna_mercado") != "comuna_otra").count()
print(f"\n📊 Gold detalle: {n_gold:,} inmuebles")
print(f"   Con barrio asignado: {n_con_barrio/n_gold*100:.1f}% (objetivo > 60%)")

display(
    df_gold.select(
        "titulo", "city_token", "comuna_mercado", "sector_mercado", "zona_mercado",
        "tipo_inmueble", "precio_num", "area_m2", "precio_m2",
        "habitaciones", "banos", "garajes", "num_portales", "dispersion_pct_grupo"
    ).limit(10)
)

# ── Resumen agregado ──
df_resumen = (
    df_gold.groupBy("city_token", "zona_mercado", "tipo_inmueble")
    .agg(
        F.count("id_original").alias("total_ofertas"),
        F.expr("percentile_approx(precio_num, 0.5)").alias("precio_mediano"),
        F.expr("percentile_approx(area_m2, 0.5)").alias("area_mediana"),
        F.expr("percentile_approx(precio_m2, 0.25)").alias("valor_m2_p25"),
        F.expr("percentile_approx(precio_m2, 0.5)").alias("valor_m2_mediano"),
        F.expr("percentile_approx(precio_m2, 0.75)").alias("valor_m2_p75"),
        F.round(F.avg("habitaciones"), 1).alias("habitaciones_promedio"),
        F.round(F.avg("banos"), 1).alias("banos_promedio"),
        F.round(F.avg(F.coalesce(F.col("num_portales"), F.lit(1))), 2).alias("promedio_portales"),
        F.round(F.avg(F.coalesce(F.col("dispersion_pct_grupo"), F.lit(0.0))), 1).alias("dispersion_promedio_pct"),
        F.round(F.avg(F.when(F.coalesce(F.col("num_portales"), F.lit(1)) > 1, F.lit(1.0)).otherwise(F.lit(0.0))) * 100, 1).alias("pct_multiportal"),
    )
    .filter(F.col("total_ofertas") >= 3)
)

In [0]:
# =============================================================
# CELDA 2: AJUSTE GEOGRÁFICO — market_token como eje comercial
# =============================================================
import os
import sys
import importlib
import importlib.util

_geo_candidates = []
try:
    _nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    _geo_candidates.append("/Workspace" + str(_nb_path).rsplit("/", 1)[0])
except Exception:
    pass
_vsc_file = globals().get("__vsc_ipynb_file__", "")
if _vsc_file:
    _geo_candidates.append(os.path.dirname(os.path.abspath(_vsc_file)))
_geo_candidates.append(os.getcwd())
for _candidate in _geo_candidates:
    if _candidate and _candidate not in sys.path:
        sys.path.insert(0, _candidate)

def _load_geo_module(module_name):
    try:
        return importlib.reload(importlib.import_module(module_name))
    except Exception as import_exc:
        last_error = import_exc
        for candidate_dir in _geo_candidates:
            if not candidate_dir:
                continue
            module_path = os.path.join(candidate_dir, f"{module_name}.py")
            if not os.path.isfile(module_path):
                continue
            try:
                spec = importlib.util.spec_from_file_location(module_name, module_path)
                module = importlib.util.module_from_spec(spec)
                spec.loader.exec_module(module)
                sys.modules[module_name] = module
                return module
            except Exception as file_exc:
                last_error = file_exc
        raise last_error

try:
    _geo_module = _load_geo_module("geography_catalog")
    GOLD_CITY_TO_MARKET = _geo_module.CITY_TO_MARKET
    print("✅ Gold usando geography_catalog.py para market_token.")
except Exception as exc:
    GOLD_CITY_TO_MARKET = {}
    print(f"⚠️ Gold no pudo cargar geography_catalog.py; usando market_token actual/fallback. Detalle: {str(exc)[:160]}")

def map_gold_market(city_token):
    city_value = normalize_text(city_token)
    if not city_value or city_value == "otra_ciudad":
        return "mercado_otro"
    return GOLD_CITY_TO_MARKET.get(city_value, "mercado_otro")

map_gold_market_udf = F.udf(map_gold_market, StringType())

def ensure_market_token(df_input):
    if "market_token" in df_input.columns:
        return (
            df_input
            .withColumn(
                "market_token",
                F.when(F.length(F.trim(F.col("market_token"))) > 0, F.col("market_token"))
                 .otherwise(map_gold_market_udf(F.col("city_token")))
            )
            .withColumn(
                "market_token",
                F.when(F.length(F.trim(F.col("market_token"))) > 0, F.col("market_token")).otherwise(F.lit("mercado_otro"))
            )
        )
    return (
        df_input
        .withColumn("market_token", map_gold_market_udf(F.col("city_token")))
        .withColumn(
            "market_token",
            F.when(F.length(F.trim(F.col("market_token"))) > 0, F.col("market_token")).otherwise(F.lit("mercado_otro"))
        )
    )

def build_market_analytics_v2(df_input, level_name, group_cols, min_obs):
    analytics = (
        df_input.groupBy(*(group_cols + ["tipo_inmueble"]))
        .agg(
            F.count("*").alias("market_n"),
            F.expr("percentile_approx(precio_num, 0.5)").alias("precio_mediano"),
            F.expr("percentile_approx(area_m2, array(0.25, 0.5, 0.75))").alias("area_quantiles"),
            F.expr("percentile_approx(precio_m2, array(0.05, 0.10, 0.25, 0.5, 0.75, 0.90, 0.95))").alias("precio_m2_quantiles"),
            F.round(F.avg(F.coalesce(F.col("num_portales"), F.lit(1))), 2).alias("promedio_portales"),
            F.round(F.avg(F.coalesce(F.col("dispersion_pct_grupo"), F.lit(0.0))), 1).alias("dispersion_promedio_pct"),
            F.round(F.avg(F.when(F.coalesce(F.col("num_portales"), F.lit(1)) > 1, F.lit(1.0)).otherwise(F.lit(0.0))) * 100, 1).alias("pct_multiportal"),
        )
        .filter(F.col("market_n") >= min_obs)
        .withColumn("analytics_level", F.lit(level_name))
    )

    if "market_token" not in group_cols:
        analytics = analytics.withColumn("market_token", F.lit("mercado_otro"))
    if "city_token" not in group_cols:
        analytics = analytics.withColumn("city_token", F.lit("otra_ciudad"))
    if "comuna_mercado" not in group_cols:
        analytics = analytics.withColumn("comuna_mercado", F.lit("comuna_otra"))
    if "sector_mercado" not in group_cols:
        analytics = analytics.withColumn("sector_mercado", F.lit("sector_otra"))

    analytics = (
        analytics
        .withColumn("area_p25", F.col("area_quantiles").getItem(0))
        .withColumn("area_mediana", F.col("area_quantiles").getItem(1))
        .withColumn("area_p75", F.col("area_quantiles").getItem(2))
        .withColumn("precio_m2_p05", F.col("precio_m2_quantiles").getItem(0))
        .withColumn("precio_m2_p10", F.col("precio_m2_quantiles").getItem(1))
        .withColumn("precio_m2_p25", F.col("precio_m2_quantiles").getItem(2))
        .withColumn("precio_m2_mediano", F.col("precio_m2_quantiles").getItem(3))
        .withColumn("precio_m2_p75", F.col("precio_m2_quantiles").getItem(4))
        .withColumn("precio_m2_p90", F.col("precio_m2_quantiles").getItem(5))
        .withColumn("precio_m2_p95", F.col("precio_m2_quantiles").getItem(6))
        .withColumn("precio_m2_iqr", F.col("precio_m2_p75") - F.col("precio_m2_p25"))
        .withColumn("lower_bound_ref", F.greatest(F.lit(0.0), F.col("precio_m2_p10") - F.col("precio_m2_iqr") * F.lit(1.5)))
        .withColumn("upper_bound_ref", F.col("precio_m2_p90") + F.col("precio_m2_iqr") * F.lit(1.5))
        .withColumn(
            "market_quality_score",
            F.round(
                F.least(
                    F.lit(100.0),
                    (F.log1p(F.col("market_n")) / F.log(F.lit(201.0))) * F.lit(60.0)
                    + F.coalesce(F.col("pct_multiportal"), F.lit(0.0)) * F.lit(0.25)
                    + (F.lit(100.0) - F.least(F.coalesce(F.col("dispersion_promedio_pct"), F.lit(100.0)), F.lit(100.0))) * F.lit(0.15),
                ),
                1,
            ),
        )
        .withColumn(
            "market_key",
            F.concat_ws("__", "analytics_level", "market_token", "city_token", "comuna_mercado", "sector_mercado", "tipo_inmueble"),
        )
        .drop("area_quantiles", "precio_m2_quantiles", "precio_m2_iqr")
    )

    return analytics.select(
        "analytics_level",
        "market_key",
        "market_token",
        "city_token",
        "comuna_mercado",
        "sector_mercado",
        "tipo_inmueble",
        "market_n",
        "precio_mediano",
        "area_p25",
        "area_mediana",
        "area_p75",
        "precio_m2_p05",
        "precio_m2_p10",
        "precio_m2_p25",
        "precio_m2_mediano",
        "precio_m2_p75",
        "precio_m2_p90",
        "precio_m2_p95",
        "lower_bound_ref",
        "upper_bound_ref",
        "promedio_portales",
        "dispersion_promedio_pct",
        "pct_multiportal",
        "market_quality_score",
    )

df_gold = ensure_market_token(df_gold).withColumn(
    "market_segment",
    F.concat_ws("__", F.col("market_token"), F.col("tipo_inmueble"))
)

# === Filtro de Cordura Geográfica (Anti-Homónimos) ===
# Correcciones para falsos 'San Vicente', 'El Retiro' o falsos listados de Medellín
df_gold = (
    df_gold
    .withColumn(
        "_ubi_lower", F.lower(F.col("ubicacion_limpia"))
    )
    .withColumn(
        "market_token",
        F.when(
            (F.col("market_token") == "oriente_antioqueno") & F.col("_ubi_lower").rlike("cali"),
            F.lit("cali_metropolitana")
        ).when(
            (F.col("market_token") == "oriente_antioqueno") & F.col("_ubi_lower").rlike("neiva"),
            F.lit("mercado_otro")
        ).when(
            (F.col("market_token") == "oriente_antioqueno") & F.col("_ubi_lower").rlike("pereira"),
            F.lit("pereira_metropolitana")
        ).when(
            (F.col("market_token") == "oriente_antioqueno") & F.col("_ubi_lower").rlike("bogot[aá]|fontib[oó]n"),
            F.lit("bogota_metropolitana")
        ).when(
            (F.col("city_token") == "medellin") & F.col("_ubi_lower").rlike("rionegro|la ceja|guarne|el retiro|marinilla|santuario"),
            F.lit("oriente_antioqueno")
        ).otherwise(F.col("market_token"))
    )
    .withColumn(
        "city_token",
        F.when(
            (F.col("market_token") == "cali_metropolitana") & F.col("_ubi_lower").rlike("cali"),
            F.lit("cali")
        ).when(
            (F.col("market_token") == "mercado_otro") & F.col("_ubi_lower").rlike("neiva"),
            F.lit("neiva")
        ).when(
            (F.col("market_token") == "pereira_metropolitana") & F.col("_ubi_lower").rlike("pereira"),
            F.lit("pereira")
        ).when(
            (F.col("market_token") == "bogota_metropolitana") & F.col("_ubi_lower").rlike("bogot[aá]|fontib[oó]n"),
            F.lit("bogota")
        ).otherwise(F.col("city_token"))
    )
    .drop("_ubi_lower")
)
# =====================================================


if df_intel is not None:
    df_intel = ensure_market_token(df_intel)

df_resumen = (
    df_gold.groupBy("market_token", "city_token", "zona_mercado", "tipo_inmueble")
    .agg(
        F.count("id_original").alias("total_ofertas"),
        F.expr("percentile_approx(precio_num, 0.5)").alias("precio_mediano"),
        F.expr("percentile_approx(area_m2, 0.5)").alias("area_mediana"),
        F.expr("percentile_approx(precio_m2, 0.25)").alias("valor_m2_p25"),
        F.expr("percentile_approx(precio_m2, 0.5)").alias("valor_m2_mediano"),
        F.expr("percentile_approx(precio_m2, 0.75)").alias("valor_m2_p75"),
        F.round(F.avg("habitaciones"), 1).alias("habitaciones_promedio"),
        F.round(F.avg("banos"), 1).alias("banos_promedio"),
        F.round(F.avg(F.coalesce(F.col("num_portales"), F.lit(1))), 2).alias("promedio_portales"),
        F.round(F.avg(F.coalesce(F.col("dispersion_pct_grupo"), F.lit(0.0))), 1).alias("dispersion_promedio_pct"),
        F.round(F.avg(F.when(F.coalesce(F.col("num_portales"), F.lit(1)) > 1, F.lit(1.0)).otherwise(F.lit(0.0))) * 100, 1).alias("pct_multiportal"),
    )
    .filter(F.col("total_ofertas") >= 3)
)

df_sectorial = (
    df_gold.groupBy("market_token", "city_token", "comuna_mercado", "sector_mercado", "tipo_inmueble")
    .agg(
        F.count("*").alias("sector_n"),
        F.expr("percentile_approx(precio_num, 0.5)").alias("precio_mediano_sector"),
        F.expr("percentile_approx(area_m2, 0.5)").alias("area_mediana_sector"),
        F.expr("percentile_approx(precio_m2, 0.10)").alias("precio_m2_p10_sector"),
        F.expr("percentile_approx(precio_m2, 0.25)").alias("precio_m2_p25_sector"),
        F.expr("percentile_approx(precio_m2, 0.5)").alias("precio_m2_mediano_sector"),
        F.expr("percentile_approx(precio_m2, 0.75)").alias("precio_m2_p75_sector"),
        F.expr("percentile_approx(precio_m2, 0.90)").alias("precio_m2_p90_sector"),
        F.round(F.avg(F.coalesce(F.col("num_portales"), F.lit(1))), 2).alias("promedio_portales"),
        F.round(F.avg(F.coalesce(F.col("dispersion_pct_grupo"), F.lit(0.0))), 1).alias("dispersion_promedio_pct"),
        F.round(F.avg(F.when(F.coalesce(F.col("num_portales"), F.lit(1)) > 1, F.lit(1.0)).otherwise(F.lit(0.0))) * 100, 1).alias("pct_multiportal"),
    )
    .filter(F.col("sector_n") >= MIN_ANALYTICS_OBS["sector"])
    .withColumn("precio_m2_min_ref", F.col("precio_m2_p10_sector"))
    .withColumn("precio_m2_max_ref", F.col("precio_m2_p90_sector"))
    .orderBy(F.desc("sector_n"), F.desc("precio_m2_mediano_sector"))
)

df_market_analytics = (
    build_market_analytics_v2(df_gold, "market", ["market_token"], max(MIN_ANALYTICS_OBS["city"], 20))
    .unionByName(build_market_analytics_v2(df_gold, "city", ["market_token", "city_token"], MIN_ANALYTICS_OBS["city"]))
    .unionByName(build_market_analytics_v2(df_gold, "comuna", ["market_token", "city_token", "comuna_mercado"], MIN_ANALYTICS_OBS["comuna"]))
    .unionByName(build_market_analytics_v2(df_gold, "sector", ["market_token", "city_token", "comuna_mercado", "sector_mercado"], MIN_ANALYTICS_OBS["sector"]))
    .orderBy(F.desc("market_quality_score"), F.desc("market_n"))
)

if df_intel is not None:
    df_oportunidades = (
        df_intel
        .withColumn("city_token", F.coalesce(F.col("city_token"), F.lit("otra_ciudad")))
        .withColumn("tipo_inmueble", F.coalesce(F.col("tipo_inmueble"), F.lit("otro")))
        .withColumn(
            "zona_mercado",
            F.coalesce(
                F.when(F.length(F.col("ubicacion_norm")) > 4, F.col("ubicacion_norm")),
                F.when(F.length(F.col("ubicacion_raw")) > 4, F.lower(F.trim(F.col("ubicacion_raw")))),
                F.concat_ws(" | ", F.col("city_token"), F.col("tipo_inmueble"))
            )
        )
        .groupBy("market_token", "city_token", "zona_mercado", "tipo_inmueble")
        .agg(
            F.count("*").alias("oportunidades_cross_portal"),
            F.round(F.avg("ahorro_potencial"), 0).alias("ahorro_promedio_cross_portal"),
            F.round(F.avg("ahorro_pct"), 1).alias("ahorro_pct_promedio"),
        )
    )
    df_resumen = df_resumen.join(df_oportunidades, ["market_token", "city_token", "zona_mercado", "tipo_inmueble"], "left")

df_resumen = df_resumen.orderBy(F.desc("valor_m2_mediano"), F.desc("total_ofertas"))

print("\n🧭 Ajuste market_token aplicado en Gold")
print(f"   market_token únicos en detalle: {df_gold.select('market_token').distinct().count():,}")
print(f"   mercado_resumen filas: {df_resumen.count():,}")
print(f"   mercado_sectorial filas: {df_sectorial.count():,}")
print(f"   mercado_analitica filas: {df_market_analytics.count():,}")
display(df_gold.select("market_token", "city_token", "comuna_mercado", "sector_mercado", "tipo_inmueble", "precio_m2").limit(20))

In [0]:
# =============================================================
# CELDA 2: ANALÍTICA OPERATIVA POR PORTAL — Checkpoints + Señales Gold
# =============================================================
from pyspark.sql import Window
from pyspark.sql.types import IntegerType, LongType, StructField, StructType, TimestampType

SILVER_PORTAL_CHECKPOINTS_PATH = config.get("silver_portal_checkpoints_path") or f"s3a://{BUCKET}/silver/portal_checkpoints/"

def load_silver_checkpoint_table(path):
    last_error = None
    for fmt in ["delta", "parquet"]:
        try:
            reader = spark.read.format(fmt)
            for k, v in S3_OPTS.items():
                reader = reader.option(k, v)
            return reader.load(path), fmt
        except Exception as exc:
            last_error = exc
    raise last_error

checkpoint_schema = StructType([
    StructField("portal", StringType(), True),
    StructField("checkpoint_key", StringType(), True),
    StructField("checkpoint_source_path", StringType(), True),
    StructField("checkpoint_activo", IntegerType(), True),
    StructField("checkpoint_last_page", LongType(), True),
    StructField("checkpoint_total_scraped", LongType(), True),
    StructField("checkpoint_updated_at", TimestampType(), True),
    StructField("checkpoint_read_error", StringType(), True),
    StructField("ingestion_timestamp", TimestampType(), True),
])

try:
    df_checkpoints_silver, checkpoint_format = load_silver_checkpoint_table(SILVER_PORTAL_CHECKPOINTS_PATH)
    checkpoint_source_label = f"{SILVER_PORTAL_CHECKPOINTS_PATH} ({checkpoint_format})"
    print(f"🛰️ Leyendo Silver checkpoints ({checkpoint_format.upper()}): {SILVER_PORTAL_CHECKPOINTS_PATH}")
except Exception as e:
    checkpoint_source_label = ""
    df_checkpoints_silver = spark.createDataFrame([], schema=checkpoint_schema)
    print(f"ℹ️ Silver checkpoints no disponible en esta corrida: {str(e)[:220]}")

for col_name, col_type in [
    ("portal", StringType()),
    ("checkpoint_key", StringType()),
    ("checkpoint_source_path", StringType()),
    ("checkpoint_activo", IntegerType()),
    ("checkpoint_last_page", LongType()),
    ("checkpoint_total_scraped", LongType()),
    ("checkpoint_updated_at", TimestampType()),
    ("checkpoint_read_error", StringType()),
    ("ingestion_timestamp", TimestampType()),
]:
    if col_name not in df_checkpoints_silver.columns:
        df_checkpoints_silver = df_checkpoints_silver.withColumn(col_name, F.lit(None).cast(col_type))

checkpoint_window = Window.partitionBy("portal").orderBy(
    F.col("checkpoint_updated_at").desc_nulls_last(),
    F.col("ingestion_timestamp").desc_nulls_last(),
    F.col("checkpoint_total_scraped").desc_nulls_last(),
)

df_checkpoints = (
    df_checkpoints_silver
    .select(
        "portal",
        "checkpoint_key",
        "checkpoint_source_path",
        "checkpoint_activo",
        "checkpoint_last_page",
        "checkpoint_total_scraped",
        "checkpoint_updated_at",
        "checkpoint_read_error",
        "ingestion_timestamp",
    )
    .withColumn("portal", F.lower(F.trim(F.col("portal"))))
    .filter(F.col("portal").isNotNull() & (F.col("portal") != ""))
    .withColumn("checkpoint_activo", F.coalesce(F.col("checkpoint_activo"), F.lit(0)))
    .withColumn("checkpoint_rank", F.row_number().over(checkpoint_window))
    .filter(F.col("checkpoint_rank") == 1)
    .drop("checkpoint_rank")
)

df_portal_operacion = (
    df_gold.groupBy("fuente")
    .agg(
        F.count("*").alias("portal_ofertas_activas"),
        F.expr("percentile_approx(precio_m2, 0.5)").alias("precio_m2_mediano_portal"),
        F.expr("percentile_approx(area_m2, 0.5)").alias("area_mediana_portal"),
        F.round(F.avg(F.coalesce(F.col("data_completeness"), F.lit(0.0))), 3).alias("data_completeness_promedio"),
        F.round(F.avg(F.coalesce(F.col("num_portales"), F.lit(1))), 2).alias("promedio_portales_portal"),
        F.round(F.avg(F.coalesce(F.col("dispersion_pct_grupo"), F.lit(0.0))), 1).alias("dispersion_promedio_portal_pct"),
        F.round(F.avg(F.when(F.coalesce(F.col("num_portales"), F.lit(1)) > 1, F.lit(1.0)).otherwise(F.lit(0.0))) * 100, 1).alias("pct_multiportal_portal"),
        F.max("fecha_extraccion").alias("ultima_fecha_extraccion"),
    )
    .withColumnRenamed("fuente", "portal")
    .join(df_checkpoints, on="portal", how="left")
    .withColumn("checkpoint_prefix_detectado", F.lit(checkpoint_source_label))
    .withColumn("checkpoint_activo", F.coalesce(F.col("checkpoint_activo"), F.lit(0)))
    .withColumn(
        "checkpoint_age_hours",
        F.when(
            F.col("checkpoint_updated_at").isNotNull(),
            F.round((F.unix_timestamp(F.current_timestamp()) - F.unix_timestamp(F.col("checkpoint_updated_at"))) / F.lit(3600.0), 1),
        )
    )
    .withColumn(
        "checkpoint_status",
        F.when(F.col("checkpoint_activo") == 0, F.lit("sin_checkpoint"))
         .when(F.col("checkpoint_updated_at").isNull(), F.lit("activo_sin_fecha"))
         .when(F.col("checkpoint_age_hours") <= 48, F.lit("activo_reciente"))
         .when(F.col("checkpoint_age_hours") <= 168, F.lit("activo_espera"))
         .otherwise(F.lit("activo_estancado"))
    )
    .withColumn(
        "portal_health_score",
        F.round(
            F.greatest(
                F.lit(0.0),
                F.least(
                    F.lit(100.0),
                    F.lit(55.0)
                    + F.least(F.log1p(F.col("portal_ofertas_activas")) * F.lit(8.0), F.lit(20.0))
                    + F.coalesce(F.col("pct_multiportal_portal"), F.lit(0.0)) * F.lit(0.10)
                    + (F.lit(100.0) - F.least(F.coalesce(F.col("dispersion_promedio_portal_pct"), F.lit(100.0)), F.lit(100.0))) * F.lit(0.10)
                    - F.when(F.col("checkpoint_activo") == 1, F.lit(15.0)).otherwise(F.lit(0.0))
                    - F.when(F.coalesce(F.col("checkpoint_age_hours"), F.lit(0.0)) > 168, F.lit(10.0)).otherwise(F.lit(0.0))
                ),
            ),
            1,
        ),
    )
    .withColumn("gold_snapshot_at", F.current_timestamp())
    .orderBy(F.asc("checkpoint_activo"), F.desc("portal_health_score"), F.desc("portal_ofertas_activas"))
)

print("\n🛰️ Analítica operativa por portal")
if checkpoint_source_label:
    print(f"   Checkpoints leídos desde Silver: {checkpoint_source_label}")
else:
    print("   No hay snapshot limpio de checkpoints en Silver. Ejecuta Silver antes de Gold.")
print(f"   Checkpoints útiles consolidados: {df_checkpoints.count():,}")
print(f"   Portales monitoreados: {df_portal_operacion.count()}")
display(df_portal_operacion.limit(100))

In [0]:
# =============================================================
# CELDA 3: GUARDAR — Gold Resumen (Delta) + Gold Sectorial (Delta) + Gold Analítica (Delta) + Gold Portal Ops (Delta) + Gold Detalle (Parquet)
# =============================================================

# Gold Resumen: agregación robusta por zona
ruta_resumen = f"s3a://{BUCKET}/gold/mercado_resumen/"
writer = df_resumen.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
for k, v in S3_OPTS.items():
    writer = writer.option(k, v)
writer.save(ruta_resumen)
print(f"💾 Gold resumen: {ruta_resumen}")

# Gold Sectorial: tabla reutilizable para analítica, app y serving
ruta_sectorial = f"s3a://{BUCKET}/gold/mercado_sectorial/"
writer = df_sectorial.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
for k, v in S3_OPTS.items():
    writer = writer.option(k, v)
writer.save(ruta_sectorial)
print(f"🧭 Gold sectorial: {ruta_sectorial}")

# Gold Analítica: bounds jerárquicos y score de calidad para limpieza y fallback
ruta_analytics = f"s3a://{BUCKET}/gold/mercado_analitica/"
writer = df_market_analytics.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
for k, v in S3_OPTS.items():
    writer = writer.option(k, v)
writer.save(ruta_analytics)
print(f"📐 Gold analítica: {ruta_analytics}")

# Gold Portal Ops: estado operativo por fuente a partir de checkpoints y snapshot Gold
ruta_portal_ops = f"s3a://{BUCKET}/gold/portal_operacion/"
writer = df_portal_operacion.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
for k, v in S3_OPTS.items():
    writer = writer.option(k, v)
writer.save(ruta_portal_ops)
print(f"🛰️ Gold portal ops: {ruta_portal_ops}")

# Gold Detalle: inmuebles limpios con KPIs robustos
ruta_app = f"s3a://{BUCKET}/gold/app_inmuebles/"
writer = df_gold.coalesce(1).write.format("parquet").mode("overwrite")
for k, v in S3_OPTS.items():
    writer = writer.option(k, v)
writer.save(ruta_app)
print(f"🚀 Gold detalle: {ruta_app}")

In [0]:
# ═══════════════════════════════════════════════════════════════
# DIAGNÓSTICO: Por qué solo el 27% tiene barrio asignado
# Correr después de que df_gold esté calculado en el Gold notebook
# ═══════════════════════════════════════════════════════════════
from pyspark.sql import functions as F

print("=" * 60)
print("1. COBERTURA DE BARRIO POR PORTAL")
print("=" * 60)
display(
    df_gold
    .groupBy("fuente")
    .agg(
        F.count("*").alias("total"),
        F.round(F.avg(
            F.when(F.col("comuna_mercado") != "comuna_otra", 1.0).otherwise(0.0)
        ) * 100, 1).alias("pct_barrio"),
        F.countDistinct("comuna_mercado").alias("comunas_unicas"),
    )
    .orderBy(F.desc("total"))
)

print("\n" + "=" * 60)
print("2. COBERTURA DE BARRIO POR CIUDAD")
print("=" * 60)
display(
    df_gold
    .groupBy("city_token")
    .agg(
        F.count("*").alias("total"),
        F.round(F.avg(
            F.when(F.col("comuna_mercado") != "comuna_otra", 1.0).otherwise(0.0)
        ) * 100, 1).alias("pct_barrio"),
        F.countDistinct("comuna_mercado").alias("comunas_unicas"),
    )
    .filter(F.col("total") >= 50)
    .orderBy(F.desc("total"))
)

print("\n" + "=" * 60)
print("3. CIUDADES CON CATÁLOGO — ubicaciones que NO matchean")
print("=" * 60)
display(
    df_gold
    .filter(
        (F.col("comuna_mercado") == "comuna_otra") &
        F.col("city_token").isin("medellin", "bogota", "cali", "barranquilla", "bucaramanga")
    )
    .groupBy("city_token", "ubicacion_limpia")
    .count()
    .orderBy(F.col("city_token"), F.desc("count"))
    .limit(40)
)

print("\n" + "=" * 60)
print("4. CIUDADES SIN CATÁLOGO — volumen que se pierde")
print("=" * 60)
ciudades_con_catalogo = ["medellin", "bogota", "cali", "barranquilla", "bucaramanga"]
display(
    df_gold
    .filter(~F.col("city_token").isin(ciudades_con_catalogo))
    .groupBy("city_token", "market_token")
    .agg(F.count("*").alias("total"))
    .filter(F.col("total") >= 100)
    .orderBy(F.desc("total"))
)

print("\n" + "=" * 60)
print("5. MUESTRA: ubicacion_limpia de cartagena, santa marta, envigado")
print("=" * 60)
display(
    df_gold
    .filter(F.col("city_token").isin("cartagena", "santa marta", "envigado", "rionegro", "pereira"))
    .groupBy("city_token", "ubicacion_limpia")
    .count()
    .orderBy(F.col("city_token"), F.desc("count"))
    .limit(40)
)